# 🪐 Jupiter und seine Monde – eine animierte OOP-Simulation

In diesem Beispiel bauen wir eine kleine, animierte Simulation des Jupitersystems. Im Kern passiert nichts Kompliziertes:

1. Jeder Himmelskörper (Jupiter, jeder Mond) ist eine **Instanz einer Klasse** und kennt seine eigenen Bahndaten.
2. Für einen gegebenen Zeitpunkt `t` berechnet jede Instanz ihre **Position** (Physik: gleichförmige Kreisbewegung).
3. Jede Instanz weiss, wie sie sich selbst **zeichnet**.
4. `matplotlib` ruft diese drei Schritte für viele aufeinanderfolgende Zeitpunkte auf – daraus entsteht die Animation.

Besonders schön: Jupiter (Zentrum) und ein Mond nutzen dieselbe Schnittstelle (`.position(t)`, `.zeichne(ax, t)`), verhalten sich aber unterschiedlich – ein greifbares Beispiel für **Polymorphismus**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# falls die Animation als eingebettetes HTML5-Video grösser als das Standardlimit wird
plt.rcParams["animation.embed_limit"] = 50  # MB

## 1. Die Klassen

- `Himmelskoerper` ist die Basisklasse: Sie kennt Name, Farbe, Grösse (fürs Zeichnen), Bahnradius (in km) und Umlaufzeit (in Tagen). Sie berechnet die Position auf einer Kreisbahn und weiss, wie sie sich zeichnet.
- `Zentralkoerper` (hier: Jupiter) überschreibt `position()` – er bewegt sich nicht, sondern bleibt im Zentrum.
- `Mond` erbt einfach die Basisklasse, ohne etwas zu ändern – eine eigene Klasse macht den Code aber lesbarer und liesse sich später erweitern (z. B. eigene Zeichenweise für unentdeckte/kleine Monde).

In [ ]:
class Himmelskoerper:
    """Ein Objekt auf einer Kreisbahn um den Ursprung (0, 0)."""

    def __init__(self, name, farbe, groesse, orbit_radius_km, umlaufzeit_tage):
        self.name = name
        self.farbe = farbe
        self.groesse = groesse
        self.orbit_radius = orbit_radius_km
        self.umlaufzeit = umlaufzeit_tage

    def position(self, t_tage):
        winkel = 2 * np.pi * t_tage / self.umlaufzeit
        x = self.orbit_radius * np.cos(winkel)
        y = self.orbit_radius * np.sin(winkel)
        return x, y

    def zeichne(self, ax, t_tage):
        x, y = self.position(t_tage)
        ax.plot(x, y, "o", color=self.farbe, markersize=self.groesse)
        # fester Pixel-Abstand (statt Daten-Koordinaten) -> Label bleibt bei jedem
        # Bahnradius lesbar, auch bei Jupiter (orbit_radius == 0)
        ax.annotate(self.name, (x, y), xytext=(0, 6 + self.groesse / 2),
                    textcoords="offset points", color=self.farbe,
                    fontsize=8, ha="center")


class Zentralkoerper(Himmelskoerper):
    """Ein Körper im Zentrum des Systems (hier: Jupiter) – bewegt sich nicht."""

    def __init__(self, name, farbe, groesse):
        super().__init__(name, farbe, groesse, orbit_radius_km=0, umlaufzeit_tage=1)

    def position(self, t_tage):
        return 0.0, 0.0


class Mond(Himmelskoerper):
    """Ein Mond, der einen Zentralkörper umkreist."""
    pass

## 2. Die Daten – Jupiter und die vier grossen ("galileischen") Monde

Reale mittlere Bahnradien (in km) und Umlaufzeiten (in Erdtagen), gerundet nach NASA/JPL-Daten. Die Verhältnisse der Abstände sind moderat genug (Faktor ~4.5 zwischen Io und Kallisto), um ohne künstliche Skalierungstricks ein gut lesbares Bild zu ergeben.

In [ ]:
jupiter = Zentralkoerper("Jupiter", farbe="orange", groesse=22)

monde = [
    Mond("Io",       farbe="gold",      groesse=6, orbit_radius_km=421_700,   umlaufzeit_tage=1.77),
    Mond("Europa",    farbe="lightblue", groesse=5, orbit_radius_km=671_100,   umlaufzeit_tage=3.55),
    Mond("Ganymed",   farbe="silver",    groesse=8, orbit_radius_km=1_070_400, umlaufzeit_tage=7.15),
    Mond("Kallisto",  farbe="peru",      groesse=7, orbit_radius_km=1_882_700, umlaufzeit_tage=16.69),
]

himmelskoerper = [jupiter] + monde

## 3. Simulation aufsetzen und animieren

Die drei Konstanten oben in der nächsten Zelle steuern Tempo und Länge der Animation – gute Stellschrauben zum Experimentieren.

In [ ]:
ZEITSCHRITT_TAGE = 0.15   # wie viele Tage vergehen pro Animationsframe
ANZAHL_FRAMES = 250       # Gesamtzahl Frames -> Gesamtzeitraum = ZEITSCHRITT_TAGE * ANZAHL_FRAMES Tage
INTERVALL_MS = 30         # Pause zwischen zwei Frames in Millisekunden

max_radius = max(mond.orbit_radius for mond in monde) * 1.15

fig, ax = plt.subplots(figsize=(6, 6))
fig.patch.set_facecolor("black")


def zeichne_hintergrund(ax):
    ax.set_facecolor("black")
    ax.set_xlim(-max_radius, max_radius)
    ax.set_ylim(-max_radius, max_radius)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for mond in monde:
        kreis = plt.Circle((0, 0), mond.orbit_radius, fill=False,
                            color="dimgray", linestyle="--", linewidth=0.6)
        ax.add_patch(kreis)


def update(frame):
    ax.clear()
    zeichne_hintergrund(ax)
    t_tage = frame * ZEITSCHRITT_TAGE
    ax.set_title(f"Jupitersystem \u2013 Tag {t_tage:.1f}", color="white")
    for koerper in himmelskoerper:
        koerper.zeichne(ax, t_tage)
    return []


anim = FuncAnimation(fig, update, frames=ANZAHL_FRAMES, interval=INTERVALL_MS, blit=False)
plt.close(fig)  # verhindert ein zusätzliches statisches Bild unter der Animation
HTML(anim.to_jshtml())

## 4. Zum Ausprobieren

- Ändert `ZEITSCHRITT_TAGE` oder `INTERVALL_MS`, um die Animation schneller/langsamer zu machen.
- Fügt der Liste `monde` einen fünften, frei erfundenen Mond hinzu.
- Blendet die gestrichelten Umlaufbahnen aus, indem ihr die Schleife in `zeichne_hintergrund` entfernt.
- Lasst jeden Mond eine kleine "Spur" (die letzten paar Positionen) zeichnen.
- **Herausforderung:** Ergänzt eine neue Klasse `Ring`, die sich wie `Zentralkoerper` verhält, aber als gestrichelter Kreis um Jupiter gezeichnet wird – ein weiteres Beispiel für Polymorphismus über dieselbe `zeichne()`-Schnittstelle.